# E0: Ava's first ears

A known architecture (QuartzNet-5x5, 6.7M parameters) trained from scratch on LibriSpeech `train-clean-100`, the ears' baseline under Zeno's Rule 1: every weight in Ava is trained by us.

Pre-registered in `docs/ears-e0.md` before this run:

- **G1, overfit sanity:** 16 training utterances, 500 steps, greedy CER ≤ 0.05 on those same utterances.
- **G2, learning sanity:** dev-clean greedy WER ≤ 0.50 for both seeds.
- **Reported, not gated:** dev-clean and test-clean WER per seed with a 95% speaker-bootstrap CI, mean and spread over two seeds, CER, epochs completed.

Every stage prints what it did and the numbers that prove it. Training prints one line per step.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

import numpy as np
import soundfile
import torch

INPUT, WORK, TEMP = Path("/kaggle/input"), Path("/kaggle/working"), Path("/kaggle/temp")
SRC, FEATURES = WORK / "src", TEMP / "e0-features.pt"
EPOCHS, MAX_HOURS = 40, 9.0

subprocess.run(["nvidia-smi"], check=False)
mem = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
print(f"python {sys.version.split()[0]}  torch {torch.__version__}", end="  ")
print(f"numpy {np.__version__}  soundfile {soundfile.__version__}")
print(f"GPUs {torch.cuda.device_count()}  CPUs {os.cpu_count()}", end="  ")
print(f"RAM {mem['MemTotal'].strip()}  available {mem['MemAvailable'].strip()}")
for d in (WORK, TEMP):
    d.mkdir(parents=True, exist_ok=True)
    print(f"free on {d}: {shutil.disk_usage(d).free / 1e9:.1f} GB", flush=True)
assert torch.cuda.device_count() >= 1, "no GPU attached"

## Inputs

The code comes from the `ava-zeno-code` dataset, uploaded by `kaggle/push_code.py` from a commit on `main`. The audio comes from the `victorling/librispeech-clean` mirror (LibriSpeech, CC BY 4.0).

In [ ]:
def mounted(slug: str) -> list[Path]:
    # Kaggle has mounted inputs at /kaggle/input/<slug> and at /kaggle/input/datasets/<owner>/<slug>.
    return sorted(p for p in [INPUT / slug, *INPUT.glob(f"datasets/*/{slug}")] if p.is_dir())


code_dir, mirror = mounted("ava-zeno-code"), mounted("librispeech-clean")
print(f"code {[str(p) for p in code_dir]}\nmirror {[str(p) for p in mirror]}")
assert len(code_dir) == 1 and len(mirror) == 1, sorted(str(p) for p in INPUT.glob("*/*/*"))
COMMIT = (code_dir[0] / "COMMIT").read_text().strip()
MIRROR = mirror[0]
shutil.copytree(code_dir[0] / "ears", SRC / "ears", dirs_exist_ok=True)
print(f"commit {COMMIT}")
print(f"copied to {SRC}: {sorted(str(p.relative_to(SRC)) for p in SRC.rglob('*.py'))}", flush=True)

## Decode the audio once

`ears.prepare` finds each subset directly and decodes every file into 64-bin log-mel features. It prints a progress line every second. Both training seeds reuse the result, so the 28,539 training files are read from network storage once.

In [ ]:
def stream(name: str, cmd: list[str], env: dict, log_path: Path) -> subprocess.Popen:
    # Prints every line a job writes, live, prefixed with the job's name, and keeps a full copy in its log file.
    p = subprocess.Popen(cmd, cwd=SRC, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    def pump() -> None:
        with log_path.open("w") as log:
            for line in p.stdout:
                log.write(line)
                log.flush()
                if line.replace("\x1b[A", "").strip():
                    print(f"[{name}] {line.rstrip()}", flush=True)

    p.pump = threading.Thread(target=pump)
    p.pump.start()
    return p


env = os.environ | {"ZENO_COMMIT": COMMIT}
t0 = time.time()
prep = stream(
    "prepare",
    [
        sys.executable,
        "-u",
        "-m",
        "ears.prepare",
        "--root",
        str(MIRROR),
        "--workers",
        str(os.cpu_count()),
        "--out",
        str(FEATURES),
    ],
    env,
    WORK / "prepare.log",
)
prep.wait()
prep.pump.join()
print(f"prepare exited {prep.returncode} after {time.time() - t0:.0f} s", flush=True)
assert prep.returncode == 0

## Train: G1 and two seeds

One job per T4. GPU 0 trains seed 0. GPU 1 first runs G1, the 16-utterance overfit check, then trains seed 1. Each job prints one line per training step, and a dev-clean evaluation every 5 epochs.

In [ ]:
common = [sys.executable, "-u", "-m", "ears.train", "--root", str(MIRROR)]
seed_args = ["--features", str(FEATURES), "--epochs", str(EPOCHS), "--max-hours", str(MAX_HOURS)]
queues = {
    0: [("e0-s0", [*common, *seed_args, "--seed", "0"])],
    1: [
        ("g1", [*common, "--overfit", "16", "--batch", "16", "--epochs", "500", "--eval-every", "100", "--seed", "0"]),
        ("e0-s1", [*common, *seed_args, "--seed", "1"]),
    ],
}
if torch.cuda.device_count() == 1:
    queues = {0: queues[1] + queues[0]}
codes = {}


def run_queue(gpu: int, jobs: list) -> None:
    for name, cmd in jobs:
        print(f"[{name}] gpu {gpu}: {' '.join(cmd)}", flush=True)
        p = stream(
            name,
            [*cmd, "--out", str(WORK / name / "ears")],
            env | {"CUDA_VISIBLE_DEVICES": str(gpu)},
            WORK / f"{name}.log",
        )
        p.wait()
        p.pump.join()
        codes[name] = p.returncode
        print(f"[{name}] exited {p.returncode}", flush=True)


t0 = time.time()
workers = [threading.Thread(target=run_queue, args=(g, jobs)) for g, jobs in queues.items()]
for w in workers:
    w.start()
for w in workers:
    w.join()
print(f"all jobs done in {(time.time() - t0) / 3600:.2f} h: exit codes {codes}", flush=True)
assert not any(codes.values()), codes

## Results against the pre-registered gates

In [ ]:
final = {n: json.loads((WORK / n / "ears" / "final.json").read_text()) for n in ("g1", "e0-s0", "e0-s1")}
g1_cer = final["g1"]["results"]["dev"]["cer"]
seeds = ["e0-s0", "e0-s1"]
print(
    f"G1 overfit: CER {g1_cer:.4f} on the 16 training utterances, bar 0.05 -> {'PASS' if g1_cer <= 0.05 else 'FAIL'}"
)
for n in seeds:
    r = final[n]["results"]
    print(
        f"{n}: {final[n]['epochs_done']} epochs  "
        f"dev WER {r['dev']['wer']:.4f} CI {np.round(r['dev']['wer_ci95'], 4).tolist()} "
        f"CER {r['dev']['cer']:.4f}  test WER {r['test']['wer']:.4f} CI {np.round(r['test']['wer_ci95'], 4).tolist()} "
        f"CER {r['test']['cer']:.4f}"
    )
dev = np.array([final[n]["results"]["dev"]["wer"] for n in seeds])
test = np.array([final[n]["results"]["test"]["wer"] for n in seeds])
g2 = bool((dev <= 0.50).all())
print(f"G2 learning: dev WER {dev.round(4).tolist()}, bar 0.50 for both seeds -> {'PASS' if g2 else 'FAIL'}")
print(
    f"two seeds: dev WER mean {dev.mean():.4f} spread {dev.max() - dev.min():.4f}; "
    f"test WER mean {test.mean():.4f} spread {test.max() - test.min():.4f}"
)
summary = {
    "commit": COMMIT,
    "G1": {"cer": g1_cer, "pass": g1_cer <= 0.05},
    "G2": {"dev_wer": dev.tolist(), "pass": g2},
    "dev_wer_mean": dev.mean(),
    "test_wer_mean": test.mean(),
    "final": final,
}
(WORK / "e0-summary.json").write_text(json.dumps(summary, indent=2))
print(f"wrote {WORK / 'e0-summary.json'}", flush=True)